<a href="https://colab.research.google.com/github/BlairRong/Python-AI-course-lab-DeepLearning/blob/main/3_9_lab__YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Unzip the dataset into the current Colab environment.

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/3.9_lab_yolo_object_detection.zip' # dataset file path
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')
print(os.listdir('/content/3.9_lab_yolo_object_detection')) # should be seeing ['images', 'labels']

['organize_images.py', 'downloaded_images', 'images', '.DS_Store', 'download_dataset.py', 'labels']


Divide the training set and validation set

In [ ]:
import os
import random
import shutil
from sklearn.model_selection import train_test_split

# setup path
base_dir = '/content/3.9_lab_yolo_object_detection'
images_dir = os.path.join(base_dir, 'images')
labels_dir = os.path.join(base_dir, 'labels')

# Create training and validation folders 创建训练和验证文件夹
train_images_dir = os.path.join(base_dir, 'train', 'images')
val_images_dir = os.path.join(base_dir, 'val', 'images')
train_labels_dir = os.path.join(base_dir, 'train', 'labels')
val_labels_dir = os.path.join(base_dir, 'val', 'labels')

for d in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
    os.makedirs(d, exist_ok=True)

# Get all image files 获取所有图片文件
image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Divide the training set and validation set 划分训练集80%和验证集20%
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

def copy_files(file_list, source_img_dir, source_label_dir, target_img_dir, target_label_dir):
    for f in file_list:
        # copy images
        src_img = os.path.join(source_img_dir, f)
        dst_img = os.path.join(target_img_dir, f)
        shutil.copy(src_img, dst_img)

        # Copy the corresponding label file (assuming the image and label file names are the same, only the suffixes are different) 复制对应的标签文件 (假设图片和标签文件名相同，只是后缀不同)
        label_file = os.path.splitext(f)[0] + '.txt'
        src_label = os.path.join(source_label_dir, label_file)
        dst_label = os.path.join(target_label_dir, label_file)
        if os.path.exists(src_label):
            shutil.copy(src_label, dst_label)
        else:
            print(f"Warning: Label file not found {src_label}")

# Perform copying 执行复制
copy_files(train_files, images_dir, labels_dir, train_images_dir, train_labels_dir)
copy_files(val_files, images_dir, labels_dir, val_images_dir, val_labels_dir)

print(f"Number of training set images: {len(os.listdir(train_images_dir))}")
print(f"Number of validation set images: {len(os.listdir(val_images_dir))}")

Number of training set images: 108
Number of validation set images: 27


Create a dataset configuration file (data.yaml) 创建数据集配置文件 (data.yaml)

In [ ]:
import yaml

data_yaml_content = """
# Paths of training and validation images 训练和验证图片的路径
path: /content/3.9_lab_yolo_object_detection  # 数据集根目录
train: train/images  # 训练图片路径 (相对于 path)
val: val/images      # 验证图片路径 (相对于 path)

# Number of categories 类别数量
nc: 3

# Category names 类别名称
names: ['red_cup', 'blue_bottle', 'phone']
"""

with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("data.yaml created successfully at /content/data.yaml")

data.yaml created successfully at /content/data.yaml


 Train YOLO Model 训练 YOLO 模型


In [ ]:
#1 Install Ultralytics YOLO

!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.7 MB/s eta 0:00:00


In [ ]:
#2.start training the YOLO model - the most important part and most spend time part

from ultralytics import YOLO

# Load the pre-trained model (using YOLOv8n, where 'n' stands for nano, which is the smallest and fastest version, suitable for my dataset and first attempt)加载预训练模型 (使用YOLOv8n，'n'代表nano，是最小最快的版本，适合我的数据集和首次尝试)
model = YOLO('yolov8n.pt')

# start training 开始训练
results = model.train(
    data='/content/data.yaml',  # Configuration file path配置文件路径
    epochs=50,                 # Training epochs: 50 epochs are sufficient for 150 images训练轮次，50轮对于150张图足够了
    imgsz=640,                 # Enter the image size, YOLOv8 defaults to 640 输入图片大小，YOLOv8默认640
    batch=16,                  # Batch size, adjusted according to Colab's video memory; 16 is fine.批次大小，根据Colab显存调整，16没问题
    patience=10,               # Early Stopping，如果10轮验证集指标没提升就停止
    device='cpu',              # Using CPU instead of GPU 使用CPU而不是GPU
    workers=2,                 # Number of data loading processes 数据加载进程数
    verbose=True               # Print detailed information 打印详细信息
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, plots=

Evaluate and Test 评估和测试

In [ ]:
# Load the trained best model 加载训练好的最佳模型
best_model = YOLO('/content/runs/detect/train/weights/best.pt') # makesure the path is correct

# Evaluate on the validation set 在验证集上评估
metrics = best_model.val(data='/content/data.yaml')
print(metrics)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1347.1±414.9 MB/s, size: 69.2 KB)
val: Scanning /content/3.9_lab_yolo_object_detection/val/labels.cache... 27 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 7.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.9s/it 5.8s
                   all         27         41      0.448      0.469      0.426      0.199
               red_cup          6         11      0.131          1      0.661      0.333
           blue_bottle         13         13      0.526      0.231      0.313      0.167
                 phone          8         17      0.686      0.176      0.304     0.0974
Speed: 2.6ms preprocess, 197.2ms inference, 0.0ms loss, 6.1ms postprocess per image
Results saved to /content/runs/detect/v

Tesk new image

In [ ]:
import shutil
import os

# Define the folder to be compressed 定义要压缩的文件夹
folder_to_zip = '/content/runs/detect/train'
# Define the output path for the zip file 定义zip文件的输出路径
output_zip_path = '/content/train_results'

# Create the zip archive 创建zip档案
shutil.make_archive(output_zip_path, 'zip', folder_to_zip)

print(f"Folder '{folder_to_zip}' has been successfully compressed to '{output_zip_path}.zip'")
print("You can now download 'train_results.zip' from the Colab file browser (left sidebar).")

Folder '/content/runs/detect/train' has been successfully compressed to '/content/train_results.zip'
You can now download 'train_results.zip' from the Colab file browser (left sidebar).


 Real-time Demo 实时演示

In [ ]:
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from IPython.display import clear_output

# Load your trained model (adjust path if necessary)
model = YOLO('/content/runs/detect/train/weights/best.pt')

# Path to your video file
video_path = '/content/drive/MyDrive/test_video.mp4'

cap = cv2.VideoCapture(video_path)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame)
    annotated = results[0].plot()
    cv2_imshow(annotated)
    clear_output(wait=True)   # refresh the displayed frame

cap.release()

results = model(frame, conf=0.1)   # show even low‑confidence detections

WARNING ⚠️ 'source' is missing. Using 'source=/usr/local/lib/python3.12/dist-packages/ultralytics/assets'.

image 1/2 /usr/local/lib/python3.12/dist-packages/ultralytics/assets/bus.jpg: 640x480 5 red_cups, 227.6ms
image 2/2 /usr/local/lib/python3.12/dist-packages/ultralytics/assets/zidane.jpg: 384x640 2 red_cups, 175.1ms
Speed: 7.0ms preprocess, 201.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
# Display the project structure of the /content/3.9_lab_yolo_object_detection folder
!ls -R /content/3.9_lab_yolo_object_detection

/content/3.9_lab_yolo_object_detection:
download_dataset.py  images  organize_images.py  val
downloaded_images    labels  train

/content/3.9_lab_yolo_object_detection/downloaded_images:
blue_bottle  phone  red_cup

/content/3.9_lab_yolo_object_detection/downloaded_images/blue_bottle:
 2c98a822475c368beb578dfa89f67e4d.png   blue_bottle_2032980.jpg
 30173d1cd57d054cd29fde1922fa3809.png   blue_bottle_2318589.jpg
 3cd0622be1d42ac2ca680be622e81397.png   blue_bottle_2408620.jpg
 45f61fe101ff891f746a4a796bdeaaa8.png   blue_bottle_2566458.jpg
 4efd3fb86e44788cd1091e19cd274f0c.png   blue_bottle_2594108.jpg
 57a2ee92c9d3ab463c06234846359178.png   blue_bottle_260019.jpg
 6c789413f4059475cbdbf97e6a2fb391.png   blue_bottle_2821977.jpg
 7585da1602906fbf657b09807330eb6d.png   blue_bottle_2886596.jpg
 8c1b7249132994cc22f40922a180f253.png   blue_bottle_3151246.jpg
 903f14ca2e522396f1f4681f8f1aa7bc.png   blue_bottle_3167368.jpg
'a blue water bottle_166406.jpg'        blue_bottle_3446015.jpg
'a blue wat

In [ ]:
import shutil
import os

# Define the main project folder to be compressed 定义要压缩的主项目文件夹
full_project_folder = '/content/3.9_lab_yolo_object_detection'
# Define the output path for the zip file 定义zip文件的输出路径
output_project_zip_path = '/content/project_archive'

# Create the zip archive 创建zip档案
shutil.make_archive(output_project_zip_path, 'zip', full_project_folder)

print(f"Full project folder '{full_project_folder}' has been successfully compressed to '{output_project_zip_path}.zip'")
print("You can now download 'project_archive.zip' from the Colab file browser (left sidebar).")

Full project folder '/content/3.9_lab_yolo_object_detection' has been successfully compressed to '/content/project_archive.zip'
You can now download 'project_archive.zip' from the Colab file browser (left sidebar).


### 下载 Colab 笔记本

要下载当前的 Colab 笔记本 (`.ipynb` 文件)，请执行以下步骤：

1.  点击 Colab 界面左上角的 **“文件”** (File) 菜单。
2.  选择 **“下载”** (Download)。
3.  选择 **“.ipynb”** 格式。

这将把您的 Colab 笔记本下载到您的本地计算机上。